In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

# STEP 2: Create sample dataset

data = {
    'Study_Hours': [2, 3, 4, 5, 6, 7, 8, 9, 1, 10,
                    3, 5, 6, 7, 8, 4, 9, 2, 10, 6],
    'Attendance': [60, 65, 70, 75, 80, 85, 90, 95, 50, 98,
                   68, 78, 82, 88, 92, 72, 96, 55, 99, 84],
    'Previous_Marks': [45, 50, 55, 60, 65, 70, 75, 80, 40, 85,
                       52, 62, 68, 72, 78, 58, 82, 42, 88, 70],
    'Assignment_Score': [50, 55, 60, 65, 70, 75, 80, 85, 45, 90,
                         58, 68, 72, 78, 82, 62, 88, 48, 92, 74],
    'Final_Marks': [35, 42, 48, 55, 63, 70, 78, 85, 30, 92,
                    45, 58, 65, 72, 80, 52, 88, 33, 95, 67]
}

df = pd.DataFrame(data)

print("Dataset:")
print(df.head())

# STEP 3: Check dataset information

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
print(df.describe())




X = df[['Study_Hours']]
y = df['Final_Marks']

# STEP 5: Split dataset into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# STEP 6: Build Simple Linear Regression model

simple_model = LinearRegression()

simple_model.fit(X_train, y_train)

# STEP 7: Make predictions

y_pred_simple = simple_model.predict(X_test)

# STEP 8: Display model coefficients

print("\nSIMPLE LINEAR REGRESSION")

print("Intercept:", simple_model.intercept_)
print("Coefficient:", simple_model.coef_[0])

# STEP 9: Evaluate model

r2_simple = r2_score(y_test, y_pred_simple)

rmse_simple = np.sqrt(
    mean_squared_error(y_test, y_pred_simple)
)

print("R2 Score:", r2_simple)
print("RMSE:", rmse_simple)

# STEP 10: Visualize Simple Linear Regression

plt.figure(figsize=(8, 5))

plt.scatter(X_test, y_test, color='blue', label='Actual Values')

plt.scatter(X_test, y_pred_simple, color='red',
            label='Predicted Values')

plt.plot(
    np.sort(X_test['Study_Hours']),
    simple_model.predict(
        pd.DataFrame({
            'Study_Hours': np.sort(X_test['Study_Hours'])
        })
    ),
    color='green',
    label='Regression Line'
)

plt.xlabel("Study Hours")
plt.ylabel("Final Marks")
plt.title("Simple Linear Regression")

plt.legend()
plt.show()


# STEP 11: Select multiple independent variables

X = df[['Study_Hours', 'Attendance',
        'Previous_Marks', 'Assignment_Score']]

y = df['Final_Marks']

# STEP 12: Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# STEP 13: Build Multiple Linear Regression model

multiple_model = LinearRegression()

multiple_model.fit(X_train, y_train)

# STEP 14: Make predictions

y_pred_multiple = multiple_model.predict(X_test)

# STEP 15: Display coefficients

print("\nMULTIPLE LINEAR REGRESSION")

print("Intercept:", multiple_model.intercept_)

print("\nCoefficients:")

for feature, coefficient in zip(
    X.columns, multiple_model.coef_
):
    print(feature, ":", coefficient)

# STEP 16: Evaluate model

r2_multiple = r2_score(y_test, y_pred_multiple)

rmse_multiple = np.sqrt(
    mean_squared_error(y_test, y_pred_multiple)
)

print("R2 Score:", r2_multiple)
print("RMSE:", rmse_multiple)




X_train_sm = sm.add_constant(X_train)

ols_model = sm.OLS(y_train, X_train_sm).fit()

print("\nREGRESSION SUMMARY")
print(ols_model.summary())


# ASSUMPTION 1: LINEARITY

plt.figure(figsize=(8, 5))

sns.scatterplot(
    x=y_pred_multiple,
    y=y_test - y_pred_multiple
)

plt.axhline(y=0, color='red', linestyle='--')

plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Linearity Check: Residual Plot")

plt.show()


# ASSUMPTION 2: NORMALITY OF RESIDUALS

residuals = y_train - ols_model.predict(X_train_sm)

plt.figure(figsize=(8, 5))

sns.histplot(residuals, kde=True)

plt.title("Normality Check: Residual Distribution")
plt.xlabel("Residuals")

plt.show()

# Q-Q Plot

sm.qqplot(residuals, line='s')

plt.title("Q-Q Plot for Normality Check")

plt.show()


# ASSUMPTION 3: HOMOSCEDASTICITY

plt.figure(figsize=(8, 5))

plt.scatter(
    ols_model.fittedvalues,
    residuals
)

plt.axhline(y=0, color='red', linestyle='--')

plt.xlabel("Fitted Values")
plt.ylabel("Residuals")

plt.title("Homoscedasticity Check")

plt.show()


# ASSUMPTION 4: MULTICOLLINEARITY (VIF)

X_vif = sm.add_constant(X_train)

vif_data = pd.DataFrame()

vif_data['Feature'] = X_vif.columns

vif_data['VIF'] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

print("\nVIF VALUES:")
print(vif_data)


# ASSUMPTION 5: INDEPENDENCE OF ERRORS

dw = durbin_watson(residuals)

print("\nDurbin Watson Statistic:", dw)


comparison = pd.DataFrame({
    'Model': ['Simple Linear Regression',
              'Multiple Linear Regression'],

    'R2 Score': [r2_simple, r2_multiple],

    'RMSE': [rmse_simple, rmse_multiple]
})

print("\nMODEL COMPARISON:")
print(comparison)


# STEP 17: Actual vs Predicted Values

plt.figure(figsize=(8, 5))

plt.scatter(
    y_test,
    y_pred_multiple,
    color='blue'
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='red',
    linestyle='--'
)

plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")

plt.title("Actual vs Predicted Values")

plt.show()


# STEP 18: Predict marks for a new student

new_student = pd.DataFrame({
    'Study_Hours': [7],
    'Attendance': [85],
    'Previous_Marks': [75],
    'Assignment_Score': [80]
})

prediction = multiple_model.predict(new_student)

print("\nPredicted Final Marks:", prediction[0])
